# Подготовка данных

Посмотрим на первые пять строк CSV. 

In [ ]:
from pathlib import Path

RENT_PATH = "data/raw/rent_v19.csv"
if Path.cwd().name == "notebooks":
    RENT_PATH = "../" + RENT_PATH

with open(RENT_PATH, encoding="utf-8-sig") as f:
    for _ in range(5):
        print(repr(f.readline()))


## 1. Формат выгрузки

- Кодировка `encoding="utf-8-sig"`
- Столбцы разделены табуляцией (`\t`).  Строки заканчиваются (`\n`).
- Дробные значения записаны через точку.
- `listing_id` читаем как строку, чтобы сохранить начальные нули. Даты записаны в формате `ДД.ММ.ГГГГ`.

In [ ]:
import pandas as pd

df = pd.read_csv(
    RENT_PATH,
    sep="\t",
    encoding="utf-8-sig",
    decimal=".",
    dtype={"listing_id": "string"},
)
df.head()


Видим сразу, что есть этажи квартиры выше, чем максимальный в доме, также в столбике расстояния до метро встречается число 99999 вероятно это обозначение высшей границы.

## 2. Разбиение данных
Удаляем точные дубли и строки без `rent_rub` и разбиваем данные. train/test = 80/20, с валидацией те же проценты.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TARGET = "rent_rub"

raw_df = pd.read_csv(
    RENT_PATH, sep="\t", encoding="utf-8-sig", dtype=str, keep_default_na=False
)
exact_duplicate = raw_df.duplicated()
missing_target = df[TARGET].isna()
eligible = df.loc[~exact_duplicate & ~missing_target]

train_val_idx, test_idx = train_test_split(
    eligible.index, test_size=0.2, random_state=RANDOM_STATE
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.2, random_state=RANDOM_STATE
)
train = df.loc[train_idx].copy()
validation = df.loc[val_idx].copy()

test_path = Path(RENT_PATH).parent.parent / "interim" / "rent_v19_test.csv"
test_path.parent.mkdir(parents=True, exist_ok=True)
raw_df.loc[test_idx].to_csv(
    test_path, sep="\t", encoding="utf-8-sig", lineterminator="\n", index=False
)

display(pd.Series({
    "Исходные строки": len(df),
    "Удалено точных дублей": exact_duplicate.sum(),
    "Удалено строк без цели после дедупликации": (~exact_duplicate & missing_target).sum(),
    "Train": len(train),
    "Validation": len(validation),
    "Test": len(test_idx),
}, name="Строк"))

## 3. Аудит качества на train

In [ ]:
audit = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing": train.isna().sum(),
    "missing_pct": train.isna().mean().mul(100),
    "unique": train.nunique(),
    "example": train.iloc[0],
})
display(audit)


In [ ]:
dates = pd.to_datetime(train["publication_date"], format="%d.%m.%Y", errors="coerce")
checks = {
    "Заполненная дата не читается": train["publication_date"].notna() & dates.isna(),
    "Год постройки позже года публикации обьявления": train["year_built"] > dates.dt.year,
    "Дата вне заданного периода": dates.notna() & ~dates.between("2026-09-01", "2026-10-31"),
    "Рейтинг вне диапазона от 1 до 5": train["owner_rating"].notna() & ~train["owner_rating"].between(1, 5),
    "Этаж выше этажности": train["floor"] > train["floors_total"],
    "Жилая площадь плюс площадь кухни больше общей": (train["living_area"] + train["kitchen_area"] - train["total_area"]).round(8) > 0,
    "Жилая площадь больше общей": train["living_area"] > train["total_area"],
    "Площадь кухни больше общей": train["kitchen_area"] > train["total_area"],
}
issue_counts = pd.Series({name: mask.sum() for name, mask in checks.items()})
display(pd.DataFrame({"rows": issue_counts, "pct_of_train": issue_counts / len(train) * 100}))

**Выводы аудита.** На train обнаружено 29 объявлений с этажом выше этажности дома и 30 объявлений с жилой площадью больше общей. У рейтинга арендодателя 3 146 пропусков из 7 777 (40,5%). Все заполненные даты разобрались и попали в заявленный период.

In [ ]:
category_candidates = [
    "district", "house_type", "elevator", "rooms", "renovation",
    "furniture", "appliances", "balcony", "parking", "pets_allowed",
    "seller_type", "max_tenants",
]
for column in category_candidates:
    counts = train[column].value_counts(dropna=False)
    display(counts.rename_axis(column).to_frame("n_rows"))

Видим, что в типе дома записано одно и то же, но разными вариантами, надо нормализовать, также в записи количества жильцов, года постройки и балконов можно дробную часть не использовать

## 4. Разведочный анализ на train

In [ ]:
iqr_columns = [
    "total_area",
    "living_area",
    "kitchen_area",
    "ceiling_height",
    "metro_distance_m",
    "distance_to_center_km",
    "response_time_hours",
]
quantitative_columns = iqr_columns + [
    "year_built", "floor", "floors_total",
]
display(train[quantitative_columns].describe().T.drop(columns=["count", "std"]))


### Выводы по описательной статистике

- **Расстояние до метро.** Значение `99999` встречается в 2 281 из 7 777 строк (29,3%). Среди остальных значений максимум — 6 000 м. Такой разрыв указывает на вероятный служебный код, который заметно завышает среднее. Перед обработкой нужно уточнить смысл этого значения.
- **Площадь и высота потолков.** Медиана общей площади — 47,2 м², максимум — 1 393 м². Для жилой площади это 26,6 и 792,3 м², для потолков — 2,67 и 6,99 м. Крайние значения стоит проверить по отдельным объявлениям перед выбором правила обработки.
- **Время ответа.** Медиана — 3 часа, 75% заполненных значений не превышают 5,9 часа, максимум — 87,4 часа. Долгий ответ возможен, поэтому такие значения стоит сначала изучить.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
def plot_rent_boxplots(ax, grouped_rent, medians, labels):
    positions = range(1, len(medians) + 1)
    values = [grouped_rent.get_group(group) for group in medians.index]
    ax.boxplot(values, orientation="horizontal", positions=positions)
    ax.set_yticks(positions, labels=labels)
    for position, median in zip(positions, medians):
        ax.annotate(
            f"{median:,.0f}".replace(",", " "), (median, position),
            xytext=(4, 0), textcoords="offset points", va="center",
            fontsize=8, bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8},
        )


fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(data=train, x=TARGET, bins=40, ax=axes[0])
axes[0].set_title("Аренда на train")
axes[0].set_xlabel("Аренда, руб./мес.")
district_rent = train.groupby("district")[TARGET]
district_medians = district_rent.median().sort_values()
plot_rent_boxplots(axes[1], district_rent, district_medians, district_medians.index)
axes[1].set(title="Аренда по районам", xlabel='Аренда, руб./мес.')
plt.tight_layout()
plt.show()

Скошено распределение целевой

In [ ]:
binary_columns = ["elevator", "appliances", "pets_allowed", "balcony"]
rent_numeric = train.select_dtypes(include="number").drop(columns=binary_columns, errors="ignore")
rooms_numeric = train["rooms"].str.extract(r"^(\d+)-", expand=False).astype(float)
rooms_numeric = rooms_numeric.mask(train["rooms"].eq("студия"), 0)

matrix_data = rent_numeric.assign(rooms_numeric=rooms_numeric)
correlation = matrix_data.corr(method="spearman")
column_order = correlation[TARGET].abs().sort_values(ascending=False).index
correlation = correlation.loc[column_order, column_order]

def plot_correlation_heatmap(correlation, title, label):
    fig, ax = plt.subplots(figsize=(13, 11), layout="constrained")
    sns.heatmap(
        correlation, annot=True, fmt=".2f", cmap="RdBu_r",
        vmin=-1, vmax=1, square=True, cbar_kws={"label": label}, ax=ax,
    )
    plt.setp(ax.get_xticklabels(), rotation=55, ha="right")
    ax.set_title(title)
    plt.show()


plot_correlation_heatmap(correlation, 'Ранговые корреляции на train', 'Корреляция Спирмена')


In [ ]:
pearson_correlation = matrix_data.corr(method="pearson").loc[column_order, column_order]
plot_correlation_heatmap(pearson_correlation, 'Корреляция Пирсона на train', 'Корреляция Пирсона')


Смотрим на корреляцию спирмана, видим, что рейтиг продавца и время ответа почти не влияет ни на один признак, признаки малоинформативны

In [ ]:
area_columns = {
    "total_area": "Общая площадь",
    "living_area": "Жилая площадь",
    "kitchen_area": "Площадь кухни",
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True, layout="constrained")
for ax, (column, title) in zip(axes, area_columns.items()):
    sns.scatterplot(data=train, x=column, y=TARGET, alpha=0.2, s=8, ax=ax)
    ax.set(ylabel="", xlabel=f"{title}, м²", title=title)
fig.supylabel('Аренда, руб./мес.')
plt.show()


Видим, что связь аренды и площади жилой зоны и общей площади одинаковы

In [ ]:
discrete_features = [
    column for column in rent_numeric.columns.drop(TARGET)
    if train[column].nunique() <= 10
]
text_features = train.select_dtypes(include=["object", "string", "category"]).columns
category_features = [
    column for column in [*text_features, *binary_columns, *discrete_features]
    if column not in {"listing_id", "publication_date", "district", "house_type"}
]
for column in category_features:
    labels = train[column].astype("string").fillna("Пропуск")
    grouped_rent = train[TARGET].groupby(labels)
    medians = grouped_rent.median().sort_values()
    ordered_labels = medians.index
    fig, ax = plt.subplots(figsize=(9, max(2.5, len(ordered_labels) * 0.34)), layout="constrained")
    tick_labels = [
        f"{label!r} (n={len(grouped_rent.get_group(label))})" for label in ordered_labels
    ]
    plot_rent_boxplots(ax, grouped_rent, medians, tick_labels)
    ax.set(title=column, xlabel='Аренда, руб./мес.')
    plt.show()


Видим, что на медиану целевой сильно влияет ремонт, колво комнат, колво жильцов. Интересно, что при 0 балконов медиана цены выше. Животные почти не влияют на целевую, как и балконы.

In [ ]:
floor_groups = pd.cut(
    train["floor"], bins=[0, 4, 9, 14, 19, float("inf")],
    labels=["1–4", "5–9", "10–14", "15–19", "20+"],
)
floor_rent_medians = (
    train.groupby([floor_groups, "balcony"], observed=True)[TARGET]
    .median()
    .unstack()
    .rename(columns=lambda count: str(int(count)))
)
ax = floor_rent_medians.plot.bar(figsize=(12, 5), rot=0)
ax.set(xlabel="Диапазон этажей", ylabel="Медианная стоимость аренды, ₽/мес.")
ax.legend(title="Число балконов")
plt.tight_layout()
plt.show()


## 5. Решения перед Pipeline
## Пропуски

In [ ]:
missing_count = train.isna().sum()
missing_count = missing_count[missing_count > 0].sort_values(ascending=False)
missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": (missing_count / len(train) * 100).round(2),
})
display(missing_summary)


1. не заполняем
2. 
3. студии
4. медиана по общей или комнатам
5. 
6. не нужен 
7. 
8.
9.
10. не заполняем
11. по комнатам медиана
12. медиана

### Закономерности пропусков на train

Посмотрим количество пропусков по типу арендодателя, по районам и в группах с заполненным или пропущенным значением связанного поля.

In [ ]:
house_mapping = {
    'панельный': 'панельный',
    'панель': 'панельный',
    'панельный дом': 'панельный',
    'панельн.': 'панельный',

    'кирпичный': 'кирпичный',
    'кирпич': 'кирпичный',
    'кирпичный дом': 'кирпичный',

    'монолитный': 'монолитный',
    'монолит': 'монолитный',
    'монолитный дом': 'монолитный',

    'блочный': 'блочный',
    'блочн.': 'блочный',
    'блочный дом': 'блочный',

    'деревянный': 'деревянный',
    'деревянный дом': 'деревянный',
    'дерево': 'деревянный',
    'деревянн.': 'деревянный'
}

house_type_normalized = train["house_type"].str.strip().str.lower()
house_type_clean = house_type_normalized.map(house_mapping).fillna(house_type_normalized)


In [ ]:
print("Пропуски в рейтинге по типу арендодателя:")
print(train.groupby("seller_type")["owner_rating"].apply(lambda x: x.isna().sum()), "\n")

print("Пропуски в высоте потолков по типу дома:")
print(train.groupby(house_type_clean, dropna=False)["ceiling_height"].apply(lambda x: x.isna().sum()), "\n")

print("Пропуски в кухонной площади по количеству комнат:")
print(train.groupby("rooms")["kitchen_area"].apply(lambda x: x.isna().sum()), "\n")



Видим, что рейтинг арендодателя не заполнен только у агенств, возможно для агенств в целом нет возможности выставить оценку. Также видим, что площадь кухни не указана только в студии, там просто нет кухни. 

## Будущий Pipeline

**Типы и категории.** `listing_id` читаем как строку, чтобы сохранить ведущие нули, и исключаем из признаков модели. Если используем `publication_date`, разбор формата `%d.%m.%Y` и извлечение признаков вынесем в Pipeline. `rooms` пока оставим категорией. Для `house_type` уберём пробелы, приведём значения к нижнему регистру и объединим известные варианты написания. `elevator`, `appliances` и `pets_allowed` оставим флагами 0/1. `balcony` и `max_tenants` оставим числовыми счётчиками.

**Пропуски.** В первой версии для `furniture`, `parking`, `renovation` и `district` используем отдельную категорию «не указано». Числовые пропуски заполним медианами train. Пропуск в `balcony` не приравниваем к нулю: ноль означает, что балкона нет. На train рейтинг отсутствует у всех агентств и заполнен у всех собственников, поэтому для первой версии заполним `owner_rating` медианой известных оценок и сохраним `seller_type`. Отдельный флаг пропуска рейтинга пока не нужен.

**Открытые вопросы.** `99999` в `metro_distance_m` похоже на служебный код. Уточним его смысл перед выбором правила. Для несогласованных этажей и площадей и для крайних значений сначала проверим примеры строк.

**Обучение предобработки.** Медианы и список категорий для кодирования будут определяться только при `fit` на train. На validation и тесте применим те же преобразования.

Мейби целевую логарифмировать и после прогноза обратно


## 6. Журнал изменений данных

Одна запись описывает одно решение. Первые две записи относятся к очистке перед разбиением. Когда добавим `Pipeline`, перед этим разделом появятся ячейки его настройки и обучения. В журнале укажем, какие преобразования применили, почему и сколько строк они затронули. Для параметров, вычисленных при `fit`, укажем, что они получены на train.

In [ ]:
cleaning_log = pd.DataFrame([
    {
        "field": "Все столбцы",
        "issue": "Точные дубли строк",
        "scope": "Исходная выгрузка",
        "n_rows": int(exact_duplicate.sum()),
        "value_before": "Повторная копия строки",
        "action": "Удалить повторные копии, оставить первое вхождение",
        "value_after": "Повторные копии исключены",
        "rationale": "Одинаковые строки не должны попадать в разные выборки",
    },
    {
        "field": TARGET,
        "issue": "Пропущена стоимость аренды",
        "scope": "Выгрузка после удаления точных дублей",
        "n_rows": int((~exact_duplicate & missing_target).sum()),
        "value_before": "Пропуск в rent_rub",
        "action": "Исключить строку",
        "value_after": "В выборках остались строки с известной стоимостью",
        "rationale": "Для обучения и оценки нужна фактическая стоимость аренды",
    },
])